In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import dill
import re

from tqdm.auto import tqdm
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans

c:\Users\ankes\.conda\envs\resume-job-analyzer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report

from sklearn.decomposition import TruncatedSVD

from sklearn.linear_model import Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from xgboost import XGBRanker

from sklearn.model_selection import GroupKFold

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
from src.features import *
from src.metric import model_evaluation

In [ ]:
with open('../data/cleaned/clean_train_df.dill','rb') as f:
    train_df=dill.load(f)
    
with open('../data/cleaned/clean_val_df.dill','rb') as f:
    val_df=dill.load(f)
        
with open('../data/cleaned/clean_test_df.dill','rb') as f:
    test_df=dill.load(f)
    


In [ ]:
def further_clean_text(text):
    # Remove special characters except spaces/./+/#
    text = re.sub(r'[^a-z0-9\s\+\#\.]', ' ', text)
        
    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    #Remove extra whitespace — always LAST
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [ ]:
train_df=train_df.apply(lambda x: further_clean_text(x))
val_df=val_df.apply(lambda x: further_clean_text(x))

test_df=test_df.apply(lambda x: further_clean_text(x))

In [ ]:
import spacy
try:
    spacy.load("en_core_web_md")
except OSError:
    print("Downloading spaCy model 'en_core_web_md'...")
    !python -m spacy download en_core_web_md --quiet
    import spacy

In [ ]:
def build_phrase_matcher(nlp, alias_dict,skills):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns=[]
    skill_map={}
    # Add alias dict
    for key, values in alias_dict.items():
        patterns.append(nlp.make_doc(key))
        skill_map[key.lower()]=key
        for v in values:
            patterns.append(nlp.make_doc(v))
            skill_map[v.lower()]=key

     # 2. Add remaining skills (from Kaggle)
    for skill in skills:
        if skill.lower() not in skill_map:
            patterns.append(nlp.make_doc(skill))
            skill_map[skill.lower()] = skill


    matcher.add("SKILLS", patterns)
    return matcher,skill_map


def single_pass_pipeline(texts, matcher,skill_map, nlp):
    processed_texts = []
    print("Applying alias normalization and phrase matching...")


    print("Processing documents with spaCy...")
    for doc in tqdm(nlp.pipe(texts, batch_size=200,n_process=2), total=len(texts)):
        raw_matches = matcher(doc)
        spans = [(match_id, start, end) for match_id, start, end in raw_matches]
        filtered_spans = filter_spans([doc[start:end] for _, start, end in spans])

        span_dict = {}
        match_map = {(start, end): match_id for match_id, start, end in spans}

        for span in filtered_spans:
            match_id = match_map[(span.start, span.end)]
            normalize_skill=skill_map.get(span.text.lower(),span.text.lower())
            span_dict[span.start] = (span.end, normalize_skill)

        tokens = []
        i = 0

        while i < len(doc):
            if i in span_dict:
                end_idx, label = span_dict[i]
                tokens.append(label.replace(" ","_"))
                i = end_idx
                continue

            token = doc[i]

            if (token.is_stop or token.is_punct or token.is_space or
                token.like_url or token.like_email):
                i += 1
                continue

            if token.pos_ not in ['NOUN', 'VERB', 'ADJ', 'PROPN']:
                i += 1
                continue

            lemma = token.lemma_.lower()

            if len(lemma) > 2 :
                tokens.append(lemma)

            i += 1

        processed_texts.append(" ".join(tokens))

    return processed_texts


def preprocess_pipeline(texts, alias_dict,skills):
    nlp = spacy.load("en_core_web_md",disable=["parser", "ner"])
    print("Building phrase matcher...")
    matcher,skill_map =build_phrase_matcher(nlp, alias_dict,skills)
    return single_pass_pipeline(texts,matcher,skill_map,nlp)


In [ ]:
resume_processed_train = preprocess_pipeline(train_df['resume_text'].tolist(), tech_aliases,skills)
job_description_processed_train = preprocess_pipeline(train_df['job_description_text'].tolist(), tech_aliases,skills)

job_description_processed_val = preprocess_pipeline(val_df['job_description_text'].tolist(), tech_aliases,skills)
resume_processed_val = preprocess_pipeline(val_df['resume_text'].tolist(), tech_aliases,skills)


resume_processed_test = preprocess_pipeline(test_df['resume_text'].tolist(), tech_aliases,skills)
job_description_processed_test = preprocess_pipeline(test_df['job_description_text'].tolist(), tech_aliases,skills)


In [ ]:
train_df['resume_clean']=resume_processed_train
train_df['jd_clean']=job_description_processed_train

val_df['resume_clean']=resume_processed_val
val_df['jd_clean']=job_description_processed_val

test_df['resume_clean']=resume_processed_test
test_df['jd_clean']=job_description_processed_test

In [ ]:
with open('../data/processed/preprocessed_train_df.dill','wb') as f:
    dill.dump(train_df,f)

with open('../data/processed/preprocessed_val_df.dill','wb') as f:
    dill.dump(val_df,f)

with open('../data/processed/preprocessed_test_df.dill','wb') as f:
    dill.dump(test_df,f)

In [ ]:
train_df.head()

In [ ]:
train_df=train_df.copy()

In [ ]:
train_df = train_df.sort_values('job_description_text').reset_index(drop=True)
val_df = val_df.sort_values('job_description_text').reset_index(drop=True)
test_df = test_df.sort_values('job_description_text').reset_index(drop=True)

In [ ]:
y_train=train_df['label']
y_val=val_df['label']

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    stop_words='english'
)

vectorizer.fit(pd.concat([train_df['resume_clean'],train_df['jd_clean']]))
X_train_resume = vectorizer.transform(train_df['resume_clean'])
X_train_jd = vectorizer.transform(train_df['jd_clean'])

X_val_resume = vectorizer.transform(val_df['resume_clean'])
X_val_jd = vectorizer.transform(val_df['jd_clean'])

X_test_resume = vectorizer.transform(test_df['resume_clean'])   
X_test_jd = vectorizer.transform(test_df['jd_clean'])

In [ ]:
train_sim = cosine_similarity(X_train_resume, X_train_jd).diagonal().reshape(-1,1)
val_sim = cosine_similarity(X_val_resume, X_val_jd).diagonal().reshape(-1,1)

test_sim = cosine_similarity(X_test_resume, X_test_jd).diagonal().reshape(-1,1)

In [ ]:
skill_overlap_train=skill_overlap(train_df['resume_clean'],train_df['jd_clean'],all_skills)
skill_overlap_val=skill_overlap(val_df['resume_clean'],val_df['jd_clean'],all_skills)

skill_overlap_test=skill_overlap(test_df['resume_clean'],test_df['jd_clean'],all_skills)

In [ ]:
exp_gap_train=train_df['resume_exp']-train_df['jd_exp']
exp_gap_val=val_df['resume_exp']-val_df['jd_exp']
exp_gap_test=test_df['resume_exp']-test_df['jd_exp']

exp_match_train=train_df['resume_exp']/(train_df['jd_exp']+1)
exp_match_val=val_df['resume_exp']/(val_df['jd_exp']+1)
exp_match_test=test_df['resume_exp']/(test_df['jd_exp']+1)

exp_enough_train=train_df['resume_exp']>=train_df['jd_exp'].astype(int)
exp_enough_val=val_df['resume_exp']>=val_df['jd_exp'].astype(int)
exp_enough_test=test_df['resume_exp']>=test_df['jd_exp'].astype(int)

In [ ]:
X_train_features = np.hstack([
    train_sim,
    skill_overlap_train,
    exp_gap_train.to_numpy().reshape(-1,1),
])

X_val_features = np.hstack([
    val_sim,
    skill_overlap_val,
    exp_gap_val.to_numpy().reshape(-1,1),
])

In [ ]:
X_test_features = np.hstack([
    test_sim,
    skill_overlap_test,
    exp_gap_test.to_numpy().reshape(-1,1),
])

In [ ]:
pd.DataFrame(X_train_features).corr()

In [ ]:
X_train_interaction = X_train_resume.multiply(X_train_jd)
X_val_interaction = X_val_resume.multiply(X_val_jd)

X_test_interaction = X_test_resume.multiply(X_test_jd)

In [ ]:
svd = TruncatedSVD(n_components=30, random_state=42)
X_train_svd = svd.fit_transform(X_train_interaction)
X_val_svd = svd.transform(X_val_interaction)
X_test_svd = svd.transform(X_test_interaction)

X_train_combine = np.hstack([X_train_svd, X_train_features])
X_val_combine = np.hstack([X_val_svd, X_val_features])
X_test_combine = np.hstack([X_test_svd, X_test_features])

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_combine)
X_val_final = scaler.transform(X_val_combine)
X_test_final = scaler.transform(X_test_combine) 


In [ ]:
models={"SVM":SVC(class_weight="balanced",probability=True),
        "LogisticRegression":LogisticRegression(class_weight="balanced",max_iter=1000),
        "RandomForest": RandomForestClassifier(n_estimators=200,class_weight="balanced", random_state=42),
        "XGB":XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1)}

In [ ]:
def train_eval_classify(X_train, X_test, y_train, y_test, model, df):
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  print("Unique predictions:", set(y_pred))
  print("Variance:", np.var(y_pred))

  y_prob = model.predict_proba(X_test)
  scores=y_prob[:,2]

  print(classification_report(y_test, y_pred))

  print("Metrics:")
  model_evaluation(scores,df,'jd_clean')

In [ ]:
for name, model_obj in models.items():
    print(f"\n--- Evaluating Model: {name} ","-"*50)
    train_eval_classify(X_train_final, X_val_final, y_train, y_val, model_obj, val_df)

In [ ]:
xgb_rank=XGBRanker(objective='rank:ndcg',learning_rate=0.05,max_depth=7, n_estimators=300,
                  subsample=0.8,colsample_bytree=0.8)

group_train=train_df.groupby('jd_clean').size().to_list()
xgb_rank.fit(X_train_final,y_train,group=group_train)
scores=xgb_rank.predict(X_val_final)

In [ ]:
print("Evaluating on Full Validation Df")
metrics=model_evaluation(scores,val_df,'jd_clean')

In [ ]:
imp_feature=xgb_rank.feature_importances_
cosine_imp=imp_feature[0]
skill_overlap_imp=imp_feature[1:4]
exp_gap_imp=imp_feature[4]
svd_imp=imp_feature[5:].sum()

In [ ]:
print("Cosine:", cosine_imp)
print("Skill_overlap:", skill_overlap_imp.sum())
print("Exp_gap:",exp_gap_imp)
print("SVD (semantic):", svd_imp)

In [ ]:
kf=GroupKFold(n_splits=5)
ndcg_scores,spearman_scores,topk_scores,mrr_scores,map_scores=[],[],[],[],[]

for fold, (train_split, val_split) in enumerate(
    kf.split(X_train_final, y_train, groups=train_df['jd_clean'])):


    xgb_rank=XGBRanker(objective='rank:ndcg',learning_rate=0.05,max_depth=7,
                       n_estimators=300,subsample=0.8,colsample_bytree=0.8)

    group_train=train_df.iloc[train_split].groupby('jd_clean').size().to_list()
    
    xgb_rank.fit(X_train_final[train_split],y_train.iloc[train_split],group=group_train)
    scores=xgb_rank.predict(X_train_final[val_split])

    print(f"for {fold+1}:\n")
    metrics=model_evaluation(scores,train_df.iloc[val_split],"jd_clean")

    spearman_scores.append(metrics['spearman_score'])
    topk_scores.append(metrics['topk_score'])
    ndcg_scores.append(metrics['ndcg_val'])
    map_scores.append(metrics['map_score'])
    mrr_scores.append(metrics['mrr_score'])
    print("-"*100)

print(f"\nCV NDCG: {np.mean(ndcg_scores):.4f} ± {np.std(ndcg_scores):.4f}")
print(f"CV Spearman: {np.mean(spearman_scores):.4f} ± {np.std(spearman_scores):.4f}")
print(f"CV Top-3 Accuracy: {np.mean(topk_scores):.4f} ± {np.std(topk_scores):.4f}")
print(f"CV MAP: {np.mean(map_scores):.4f} ± {np.std(map_scores):.4f}")
print(f"CV MRR: {np.mean(mrr_scores):.4f} ± {np.std(mrr_scores):.4f}")



In [ ]:
final_score=xgb_rank.predict(X_test_final)
metrics=model_evaluation(final_score,test_df,'jd_clean')